# Non-SNP-only candidate genes — what does kMate's indel+SV layer flag that SNPs miss?

For every contrast we ran (multitrait **JOINT**, **GLOBAL**, **CLIMATE** × [bio1–19 + PC1 of all
bioclim], and 31 **per-site** scans), we took the clq0.9 LD blocks BH-FDR-significant in the
**non-SNP (indel+SV)** GWAS but **not** in the SNP GWAS — the peaks a SNP-only GWAS would miss —
then mapped each block to overlapping **TAIR10 genes** (within the block span, plus a **±2 kb
promoter flank** to catch nearby regulatory targets) and described each gene via the **Ensembl
Plants** and **UniProt** REST APIs.

**Caveat up front (read before interpreting):** these are, by construction, the *fragile* end of
the signal — blocks one marker class calls significant and the other doesn't. The variance-
partition work showed non-SNP adds ~nothing genome-wide; this list is the scattered local
exceptions. Most are FDR-level (not Bonferroni), and the CLIMATE-contrast subset attaches to
bioclim hits that are likely FDR-tail noise (they mostly vanish under Bonferroni, don't replicate
across classes, and aren't corrected across the 20 collinear climate axes). Treat as a hypothesis-
generating candidate list, not confirmed kMate-unique adaptation loci.

In [1]:

import os
import numpy as np, pandas as pd
os.chdir("/global/scratch/users/tbellg/kmate")
OUT = "analysis/grenenet_gea/varexp"
g = pd.read_csv(f"{OUT}/nonsnp_only_genes_described.csv").fillna("")
b = pd.read_csv(f"{OUT}/nonsnp_only_blocks.csv").fillna("")
g["desc"] = g["ensembl_description"].str.replace(r" \[Source:.*", "", regex=True)

def ctype(s):
    ks = set()
    for c in s.split(";"):
        if c.startswith("multitrait_CLIMATE"): ks.add("CLIMATE")
        elif c.startswith("multitrait_JOINT"): ks.add("JOINT")
        elif c.startswith("multitrait_GLOBAL"): ks.add("GLOBAL")
        elif c.startswith("persite"): ks.add("per-site")
    return ",".join(sorted(ks))
g["contrast_types"] = g["contrasts"].apply(ctype)
print(f"{len(g)} non-SNP-only candidate genes across {len(b)} blocks")
print(f"  in-block: {(g.overlap_type=='in_block').sum()}   +/-2kb flank-only: {(g.overlap_type=='flank2kb').sum()}")
print(f"  Bonferroni-subset genes: {g.bonferroni.sum()}")
print(f"  with an Ensembl description: {(g.ensembl_description!='').sum()};  with a UniProt function: {(g.uniprot_function!='').sum()}")


969 non-SNP-only candidate genes across 431 blocks
  in-block: 465   +/-2kb flank-only: 504
  Bonferroni-subset genes: 71
  with an Ensembl description: 945;  with a UniProt function: 442


## Bonferroni-stricter subset — the most defensible non-SNP-only hits

Genes under blocks that are non-SNP-only at the **block-level Bonferroni** threshold (not just
FDR) in at least one contrast. This is the subset that survives the strict multiple-testing bar.

In [2]:

bonf = g[g.bonferroni].copy().sort_values(["contrast_types", "gene"])
with pd.option_context("display.max_colwidth", 80, "display.width", 240):
    print(bonf[["gene", "symbol", "desc", "overlap_type", "contrast_types", "contrasts"]].to_string(index=False))


     gene    symbol                                                                                      desc overlap_type         contrast_types                                                                                                                                                                           contrasts
AT3G52800 AT3G52800                                                   A20/AN1-like zinc finger family protein     flank2kb                CLIMATE                                                                                                                                    multitrait_CLIMATE_bio1;multitrait_CLIMATE_bio10
AT3G52810     PAP21                                                                purple acid phosphatase 21     in_block                CLIMATE                                                                                                                                    multitrait_CLIMATE_bio1;multitrait_CLIMATE_bio10
AT3G52820     PAP22   

## Themed screen — flowering time / cold / heat / circadian

Keyword match over each gene's symbol + Ensembl description + UniProt function text. This is
**high-precision / low-recall**: it catches genes whose annotation *explicitly* names the process,
but a gene with only a generic family name (e.g. "NAC domain protein") won't match even if it is
in fact stress-related. So absence here is not evidence of absence — see the manual notes below.

In [3]:

THEMES = {
    "flowering_time": ["flowering", "floral", "vernaliz", "photoperiod", "inflorescence",
                       "constans", "gigantea", "frigida", "agamous", "apetala", "flc", "soc1",
                       " ft ", "mads", "meristem identity"],
    "cold": ["cold", "freezing", "chilling", "cbf", "dreb", "cor15", "low temperature",
             "ice1", "dehydrin", "cold acclimation", "cold-regulated", "cold regulated"],
    "heat": ["heat shock", "heat stress", "high temperature", "thermotoler", "thermomorph",
             "hsp", "hsf", "chaperone"],
    "circadian": ["circadian", "clock", "cca1", "lhy", "toc1", "pseudo-response regulator",
                  "zeitlupe", "rhythm", " prr", "elf3", "elf4"],
}
text = (g["symbol"].str.lower() + " | " + g["desc"].str.lower() + " | " + g["uniprot_function"].str.lower())
for th, kws in THEMES.items():
    g[th] = text.apply(lambda t: any(k in t for k in kws))
g["themes"] = g.apply(lambda r: ",".join(th for th in THEMES if r[th]), axis=1)
hit = g[g["themes"] != ""]
print(f"{len(hit)} genes matched a theme keyword:")
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    print(hit[["gene", "symbol", "themes", "desc", "contrast_types"]].to_string(index=False))


40 genes matched a theme keyword:
     gene     symbol                             themes                                                                                                           desc contrast_types
AT4G21870  AT4G21870                               heat                                                                      HSP20-like chaperones superfamily protein       per-site
AT5G06170       SUC9                     flowering_time                                                                                     sucrose-proton symporter 9       per-site
AT5G49060  AT5G49060                               heat                                                        DnaJ heat shock amino-terminal domain protein (DUF1977)       per-site
AT2G37390      NAKR2                               heat                                                                  Chloroplast-targeted copper chaperone protein       per-site
AT3G14380  AT3G14380                     flowering_time 

### UniProt function text for the theme-matched genes (fuller context)

In [4]:

with pd.option_context("display.max_colwidth", 200, "display.width", 260):
    for _, r in hit.iterrows():
        fn = r["uniprot_function"] or "(no UniProt function annotation)"
        print(f"- {r['gene']} ({r['symbol'] or r['desc']}) [{r['themes']}]: {fn}\n")


- AT4G21870 (AT4G21870) [heat]: (no UniProt function annotation)

- AT5G06170 (SUC9) [flowering_time]: High-affinity sucrose transporter. Responsible for the transport of sucrose into the cell, with the concomitant uptake of protons (symport system). Can also transport a wide range of glucosides, such as helicin, salicin, arbutin, maltose, fraxin, esculin, uranose, alpha-methylglucoside, alpha-phenylglucoside and beta-phenylglucoside. Plays a role in flowering time transition delay. {ECO:0000269|PubMed:15361146, ECO:0000269|PubMed:17098854}.

- AT5G49060 (AT5G49060) [heat]: Plays a continuous role in plant development probably in the structural organization of compartments. {ECO:0000250}.

- AT2G37390 (NAKR2) [heat]: (no UniProt function annotation)

- AT3G14380 (AT3G14380) [flowering_time]: Involved in floral organ shedding. {ECO:0000269|PubMed:22992509}.

- AT5G62940 (HCA2) [flowering_time]: Transcription factor that binds specifically to a 5'-AA[AG]G-3' consensus core sequence (By s

## Full annotated table (all 126 genes)

Sorted Bonferroni-first, then by number of contrasts. `overlap_type` = in_block vs ±2kb flank.

In [5]:

show = g.sort_values(["bonferroni", "n_contrasts", "gene"], ascending=[False, False, True])
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60, "display.width", 260):
    print(show[["gene", "symbol", "desc", "biotype", "overlap_type", "bonferroni",
                "n_contrasts", "contrast_types"]].to_string(index=False))


     gene     symbol                                                                                                           desc            biotype overlap_type  bonferroni  n_contrasts         contrast_types
AT5G04160  AT5G04160                                                                    Nucleotide-sugar transporter family protein     protein_coding     flank2kb        True           12               per-site
AT5G04170  AT5G04170                                                                         Calcium-binding EF-hand family protein     protein_coding     in_block        True           12               per-site
AT5G04180       ACA3                                                                                     alpha carbonic anhydrase 3     protein_coding     in_block        True           12               per-site
AT3G22820       CLL1                                                                                          allergen-like protein     protein_coding  

## Notes

- The **JOINT** (any-site selection) genes are the strongest-motivated subset (JOINT is the one
  contrast with a real, well-calibrated genome-wide signal); CLIMATE-contrast genes inherit the
  bioclim-null caveat above.
- Keyword screening is low-recall. A manual pass over the symbols (below the auto-screen) is worth
  doing for canonical stress/flowering/clock genes that carry only generic family annotations.
- Files: `analysis/grenenet_gea/varexp/nonsnp_only_{blocks,genes,genes_described}.csv`.